## Bivariate Analysis

**Information Need:** Assess the statistical dependence between two selected event log attributes.

**Motivation:** Assess the statistical dependence between two selected attributes based on their jointly observed values using a suitable dependence measure, such as the normalized mutual information, or by means of graphical exploration, for example by inspecting a scatterplot.

**Approach:** Assess the statistical dependence between two selected attributes based on their jointly observed values using a suitable dependence measure or by means of graphical exploration, for example by inspecting a scatterplot.

**Output:** An assessment of the presence, strength, and where present form of statistical dependence between the selected attributes, based on the applied dependence measure or graphical exploration.

In [ ]:
import pandas as pd
import numpy as np
import pm4py

from ipywidgets import interact, IntSlider

import matplotlib.pyplot as plt


# --- Configuration -----------------------------------------------------------
LOG_PATH = "../../data/BPI_Challenge_2017.xes"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)


display(event_log.head())

### Pattern execution

In [ ]:
columns = event_log.columns.tolist()

def discretize(series, bins):
    if pd.api.types.is_numeric_dtype(series):
        return pd.qcut(series, bins, duplicates='drop')
    return series


def entropy(series):
    probabilities = series.dropna().value_counts(normalize=True)
    return -(probabilities * np.log2(probabilities)).sum()


def mutual_information(x, y):
    joint = pd.crosstab(x, y, normalize=True)
    px = joint.sum(axis=1)
    py = joint.sum(axis=0)
    mi = 0.0
    for i in joint.index:
        for j in joint.columns:
            p_xy = joint.loc[i, j]
            if p_xy > 0:
                mi += p_xy * np.log2(p_xy / (px[i] * py[j]))
    return mi


def normalized_mutual_information(x, y, bins):
    mask = x.notna() & y.notna()
    if not mask.any():
        return None  # no jointly-populated rows

    x_binned = discretize(x[mask], bins)
    y_binned = discretize(y[mask], bins)

    hx = entropy(x_binned)
    hy = entropy(y_binned)
    mi = mutual_information(x_binned, y_binned)

    denom = (hx + hy) / 2
    nmi = mi / denom if denom > 0 else 0.0

    return hx, hy, mi, nmi

In [ ]:
@interact(attribute_x=columns, attribute_y=columns, bins=IntSlider(min=2, max=20, value=10, description='bins'))
def assess_dependence(attribute_x, attribute_y, bins=10):
    if attribute_x == attribute_y:
        print(f'{attribute_x} is compared to itself')
        return

    result = normalized_mutual_information(event_log[attribute_x], event_log[attribute_y], bins)
    if result is None:
        print(f'No rows where both {attribute_x} and {attribute_y} are populated')
        return

    hx, hy, mi, nmi = result
    print(f'H({attribute_x}) = {hx:.4f} bits')
    print(f'H({attribute_y}) = {hy:.4f} bits')
    print(f'MI({attribute_x}; {attribute_y}) = {mi:.4f} bits')
    print(f'NMI({attribute_x}; {attribute_y}) = {nmi:.4f}')

### Graphical exploration

In [ ]:
numeric_columns = event_log.select_dtypes(include='number').columns.tolist()

@interact(
    attribute_x=numeric_columns,
    attribute_y=numeric_columns
)
def plot_attribute_dependence(attribute_x, attribute_y):

    if attribute_x == attribute_y:
        print(f'{attribute_x} is plotted against itself')

    # Keep only jointly observed values
    plot_data = event_log[[attribute_x, attribute_y]].dropna()

    if plot_data.empty:
        print(
            f'No rows where both {attribute_x} and '
            f'{attribute_y} are populated'
        )
        return

    plt.figure(figsize=(8, 5))

    plt.scatter(
        plot_data[attribute_x],
        plot_data[attribute_y],
        alpha=0.2, #change as needed
        s=10 #change as needed
    )

    plt.xlabel(attribute_x)
    plt.ylabel(attribute_y)
    plt.title(
        f'{attribute_x} vs. {attribute_y}\n'
        f'n = {len(plot_data):,} jointly observed events'
    )

    plt.grid(alpha=0.2)
    plt.show()